# Fast Inference with KV Cache, Flash Attention, and Speculative Decoding

Training a model is a one-time cost. Inference is paid on every request, forever. A model that generates 10 tokens/sec is not deployable; a model that generates 500 tokens/sec can serve real users. The difference is almost entirely algorithmic — the same weights, the same hardware, orders of magnitude apart in throughput. This notebook covers the three techniques that make the difference: [**KV caching**]{.mark}, which eliminates redundant recomputation during autoregressive generation; **Flash Attention**, which eliminates redundant HBM traffic during the attention computation; and **speculative decoding**, which breaks the sequential bottleneck by verifying multiple draft tokens in a single parallel forward pass.

## The Autoregressive Generation Problem

Language model generation is sequential: to generate token $t+1$, you need token $t$. The naive forward pass for generating token $t+1$ runs on the full accumulated sequence $[x_1, \ldots, x_t]$ — recomputing all keys and values from scratch. Cost per token: $O(t)$ attention operations. Cost to generate $T$ tokens: $O(T^2)$. For $T = 1024$, that is one million units of work where one thousand would suffice.

The fix is to **cache** the computed keys and values. After computing $K_l$ and $V_l$ for layer $l$ at position $t$, store them. At step $t+1$, only compute the new query $q_{t+1}$ and the new key/value pair $(k_{t+1}, v_{t+1})$. Attend to the full cached sequence without recomputing anything.

## KV Cache

The KV cache stores key and value projections for all past tokens across all layers. Two phases:

- **Prefill:** run the full prompt through the model in one forward pass, populating the cache for all prompt positions.
- **Decode:** generate one token at a time, computing only the new key/value pair and attending to the full cache.

Implementing `KVCache`:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class KVCache:
    """Stores key and value tensors for all transformer layers.

    On each generation step, new K/V pairs are appended along the
    sequence dimension. Shape: k_cache[layer] = (B, H, T_cached, d_h).

    Args:
        n_layers: number of transformer layers.
    """

    n_layers: int
    k_cache: list[Optional[torch.Tensor]] = field(default_factory=list)
    v_cache: list[Optional[torch.Tensor]] = field(default_factory=list)
    seq_len: int = 0

    def __post_init__(self) -> None:
        self.k_cache = [None] * self.n_layers
        self.v_cache = [None] * self.n_layers

    def update(
        self,
        layer_idx: int,
        new_k: torch.Tensor,
        new_v: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Append new K/V to the cache and return the full cached tensors.

        Args:
            layer_idx: which layer's cache to update.
            new_k: shape (B, H, T_new, d_h).
            new_v: shape (B, H, T_new, d_h).

        Returns:
            (k_full, v_full): full cached K/V tensors including the new tokens.
        """
        if self.k_cache[layer_idx] is None:
            self.k_cache[layer_idx] = new_k
            self.v_cache[layer_idx] = new_v
        else:
            self.k_cache[layer_idx] = torch.cat([self.k_cache[layer_idx], new_k], dim=2)
            self.v_cache[layer_idx] = torch.cat([self.v_cache[layer_idx], new_v], dim=2)

        if layer_idx == 0:
            self.seq_len = self.k_cache[0].size(2)

        return self.k_cache[layer_idx], self.v_cache[layer_idx]

    def clear(self) -> None:
        """Reset the cache between independent sequences."""
        self.k_cache = [None] * self.n_layers
        self.v_cache = [None] * self.n_layers
        self.seq_len = 0

    def memory_bytes(self) -> int:
        """Current cache memory usage in bytes."""
        total = 0
        for k, v in zip(self.k_cache, self.v_cache):
            if k is not None:
                total += k.numel() * k.element_size()
                total += v.numel() * v.element_size()
        return total

**Cache-aware attention.** The attention module needs a code path that: (1) computes K and V only for the new token(s), (2) reads the full K/V from the cache, and (3) computes attention between the new query and all cached K/V. During decoding the query has shape `(B, H, 1, d_h)` — a single vector attending over the entire cached sequence.

Implementing `attention_with_cache` and the two-phase generation loop:

In [ ]:
def attention_with_cache(
    q: torch.Tensor,
    k_new: torch.Tensor,
    v_new: torch.Tensor,
    cache: KVCache,
    layer_idx: int,
    is_causal: bool = True,
) -> torch.Tensor:
    """Attention computation using the KV cache.

    For generation (T_new=1), this reduces to a single vector-matrix product.

    Args:
        q: shape (B, H, T_new, d_h) — query for new token(s).
        k_new: shape (B, H, T_new, d_h) — new key(s) to append.
        v_new: shape (B, H, T_new, d_h) — new value(s) to append.
        cache: KVCache object holding past K/V.
        layer_idx: which layer this call belongs to.
        is_causal: apply causal masking (only needed during prefill).

    Returns:
        Output tensor of shape (B, H, T_new, d_h).
    """
    k_full, v_full = cache.update(layer_idx, k_new, v_new)  # <1>
    d_h = q.size(-1)
    scale = d_h ** -0.5
    scores = torch.matmul(q, k_full.transpose(-2, -1)) * scale  # (B, H, T_new, T_cached)

    # Causal mask only needed during prefill (T_new > 1)
    if is_causal and q.size(2) > 1:
        T_new = q.size(2)
        T_cached = k_full.size(2)
        mask = torch.ones(T_new, T_cached, device=q.device, dtype=torch.bool)
        mask = torch.triu(mask, diagonal=T_cached - T_new + 1)
        scores = scores.masked_fill(mask.unsqueeze(0).unsqueeze(0), -1e9)

    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, v_full)  # (B, H, T_new, d_h)


def generate_with_cache(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 200,
    temperature: float = 0.8,
    top_k: int = 50,
    device: torch.device = None,
) -> str:
    """Autoregressive generation with KV cache.

    Two phases:
    1. Prefill: full prompt in one forward pass, populating the cache.
    2. Decode: one token at a time using the cache.

    Args:
        model: language model with KV cache support.
        tokenizer: tokenizer with encode/decode methods.
        prompt: input prompt string.
        max_new_tokens: maximum tokens to generate.
        temperature: sampling temperature.
        top_k: top-k sampling parameter.
        device: compute device.

    Returns:
        Generated text string (excluding prompt).
    """
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    cache = KVCache(n_layers=model.config.n_layers)
    prompt_ids = torch.tensor(
        [tokenizer.encode(prompt)], dtype=torch.long, device=device
    )

    generated = []

    with torch.no_grad():
        # Phase 1: prefill
        logits = model(prompt_ids, cache=cache)   # <2>
        next_token_logits = logits[0, -1, :]      # last position

        # Phase 2: decode
        for _ in range(max_new_tokens):
            if temperature > 0:
                scaled = next_token_logits / temperature
                if top_k > 0:
                    topk_vals, _ = torch.topk(scaled, top_k)
                    scaled[scaled < topk_vals[-1]] = -float("inf")
                probs = F.softmax(scaled, dim=-1)
                next_id = torch.multinomial(probs, num_samples=1)
            else:
                next_id = next_token_logits.argmax(dim=-1, keepdim=True)

            generated.append(next_id.item())
            if next_id.item() == tokenizer.eos_id:
                break

            # Single-token forward pass
            logits = model(next_id.unsqueeze(0), cache=cache)  # <3>
            next_token_logits = logits[0, -1, :]

    return tokenizer.decode(generated)

1. `cache.update` appends the new K/V and returns the full cached tensor — the attention scores are computed between the new query and all past + present keys.
2. Prefill: full prompt sequence, populates the cache with K/V for all prompt positions.
3. Decode: single new token as input; the model reads the accumulated cache for context.

### Cache memory

The KV cache memory scales linearly with sequence length:

$$\text{memory} = 2 \times L \times H \times d_h \times T \times \text{bytes per element}$$

For a 7B model (32 layers, 32 heads, $d_h = 128$) at FP16 with $T = 4096$: $2 \times 32 \times 32 \times 128 \times 4096 \times 2 \approx 2$ GB. This is why KV cache quantization (INT8 or INT4 keys/values) is important for long-context serving.

### Cache eviction strategies

When the generated sequence exceeds the maximum context length, the cache must be truncated. Two common strategies:

In [ ]:
class SlidingWindowEviction:
    """Keep only the most recent max_len tokens.

    Simple and effective for most chat use cases.
    The model loses context beyond max_len tokens.
    """

    def __call__(self, cache: KVCache, max_len: int) -> None:
        """Evict oldest tokens to bring seq_len down to max_len.

        Args:
            cache: KVCache to evict from.
            max_len: target maximum sequence length.
        """
        if cache.seq_len <= max_len:
            return
        for i in range(cache.n_layers):
            if cache.k_cache[i] is not None:
                cache.k_cache[i] = cache.k_cache[i][:, :, -max_len:, :]
                cache.v_cache[i] = cache.v_cache[i][:, :, -max_len:, :]
        cache.seq_len = max_len


class SinkTokenEviction:
    """StreamingLLM (Xiao et al. 2023) eviction strategy.

    Always keep the first n_sink tokens (attention sinks) plus the
    most recent (max_len - n_sink) tokens. Preserves the initial context
    that anchors the model's attention patterns.

    Args:
        n_sink: number of initial tokens to always preserve.
    """

    def __init__(self, n_sink: int = 4) -> None:
        self.n_sink = n_sink

    def __call__(self, cache: KVCache, max_len: int) -> None:
        """Evict tokens from the middle of the cache.

        Args:
            cache: KVCache to evict from.
            max_len: target maximum sequence length.
        """
        if cache.seq_len <= max_len:
            return
        n_drop = cache.seq_len - max_len
        n_keep_recent = max_len - self.n_sink

        for i in range(cache.n_layers):
            if cache.k_cache[i] is None:
                continue
            k, v = cache.k_cache[i], cache.v_cache[i]
            sink_k = k[:, :, :self.n_sink, :]
            sink_v = v[:, :, :self.n_sink, :]
            recent_k = k[:, :, self.n_sink + n_drop:, :]
            recent_v = v[:, :, self.n_sink + n_drop:, :]
            cache.k_cache[i] = torch.cat([sink_k, recent_k], dim=2)
            cache.v_cache[i] = torch.cat([sink_v, recent_v], dim=2)

        cache.seq_len -= n_drop

## Flash Attention

Standard attention is IO-bound, not compute-bound. For a sequence of length $T$, the naive implementation:

1. Computes $S = QK^\top / \sqrt{d_h}$ — shape $(T, T)$.
2. Writes $S$ to HBM (GPU main memory).
3. Reads $S$ from HBM to compute softmax.
4. Writes softmax weights $P$ to HBM.
5. Reads $P$ from HBM to compute $PV$.

Steps 2–4 each read/write an $O(T^2)$ matrix. At $T = 2048$: ~8 MB per read/write per layer. HBM bandwidth on an A100 is 2 TB/s; on-chip SRAM bandwidth is ~20 TB/s. [The bottleneck is the HBM round-trips, not the FLOPs.]{.underline}

### Flash Attention: tiling to stay in SRAM

Flash Attention [@dao2022flashattention] rewrites the attention computation so that intermediate matrices never need to be written to HBM. It processes attention in tiles that fit in SRAM, using the **online softmax** trick to accumulate the output without materializing the full $T \times T$ matrix.

**Online softmax.** For a vector $x \in \mathbb{R}^T$, standard softmax requires two passes (find maximum, then compute exponentials). The online algorithm processes one element at a time, maintaining a running maximum $m$, normalizer $l$, and output $o$:

```
m_0 = -∞,  l_0 = 0,  o_0 = 0
For i = 1 to T:
    m_i = max(m_{i-1}, x_i)
    l_i = l_{i-1} * exp(m_{i-1} - m_i) + exp(x_i - m_i)
    o_i = o_{i-1} * exp(m_{i-1} - m_i) / l_i * l_{i-1} + exp(x_i - m_i) / l_i * v_i
```

After the full pass, $o_T = \text{softmax}(x) \cdot V$ — the attention output — without ever storing the $T$-length attention weights.

A reference implementation in pure PyTorch (for illustration — the real version uses custom CUDA kernels):

In [ ]:
def flash_attention_reference(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    block_size: int = 64,
    causal: bool = True,
) -> torch.Tensor:
    """Reference implementation of Flash Attention in pure PyTorch.

    This is NOT as fast as the real CUDA implementation — it illustrates
    the tiling algorithm. For production use, call
    F.scaled_dot_product_attention() which uses the optimized kernel.

    Args:
        Q: shape (B, H, T, d_h).
        K: shape (B, H, T, d_h).
        V: shape (B, H, T, d_h).
        block_size: tile size for SRAM blocking.
        causal: apply causal masking.

    Returns:
        Output tensor of shape (B, H, T, d_h).
    """
    B, H, T, d_h = Q.shape
    scale = d_h ** -0.5
    O = torch.zeros_like(Q)

    for q_start in range(0, T, block_size):
        q_end = min(q_start + block_size, T)
        Q_blk = Q[:, :, q_start:q_end, :]   # (B, H, Bq, d_h)
        Bq = q_end - q_start

        m_i = torch.full((B, H, Bq), float("-inf"), device=Q.device)  # running max
        l_i = torch.zeros(B, H, Bq, device=Q.device)                  # running sum
        O_i = torch.zeros(B, H, Bq, d_h, device=Q.device)             # running output

        for kv_start in range(0, T, block_size):
            kv_end = min(kv_start + block_size, T)
            if causal and kv_start >= q_end:  # <1>
                break

            K_blk = K[:, :, kv_start:kv_end, :]
            V_blk = V[:, :, kv_start:kv_end, :]

            S_blk = torch.matmul(Q_blk, K_blk.transpose(-2, -1)) * scale  # (B, H, Bq, Bkv)

            if causal:
                q_idx = torch.arange(q_start, q_end, device=Q.device)
                kv_idx = torch.arange(kv_start, kv_end, device=Q.device)
                causal_mask = q_idx.unsqueeze(1) < kv_idx.unsqueeze(0)   # (Bq, Bkv)
                S_blk = S_blk.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), -1e9)

            # Online softmax update
            m_blk = S_blk.max(dim=-1).values              # (B, H, Bq)
            m_new = torch.maximum(m_i, m_blk)             # <2>
            exp_S = torch.exp(S_blk - m_new.unsqueeze(-1))
            exp_correction = torch.exp(m_i - m_new)       # <3>

            l_new = l_i * exp_correction + exp_S.sum(dim=-1)
            O_i = (O_i * (l_i * exp_correction).unsqueeze(-1)
                   + torch.matmul(exp_S, V_blk)) / l_new.unsqueeze(-1)

            m_i, l_i = m_new, l_new

        O[:, :, q_start:q_end, :] = O_i

    return O

1. Causal masking: entire future KV blocks can be skipped without computation — causal attention only attends to positions $\leq$ current position.
2. Running maximum update: when a new block with a larger value arrives, all previous exponentials must be rescaled.
3. Correction factor: multiply past running sum and output by $\exp(m_{\text{old}} - m_{\text{new}})$ to account for the updated normalization.

### Replacing standard attention with Flash Attention

The change to the model is a single line. `F.scaled_dot_product_attention` (PyTorch ≥ 2.0) calls the optimized Flash Attention kernel automatically:

In [ ]:
# In MultiHeadAttention.forward():

# BEFORE (standard attention — O(T^2) HBM traffic):
scores = torch.matmul(Q, K.transpose(-2, -1)) * scale
scores = scores.masked_fill(causal_mask, -1e9)
weights = F.softmax(scores, dim=-1)
output = torch.matmul(weights, V)

# AFTER (Flash Attention — O(T) memory, same result):
output = F.scaled_dot_product_attention(
    Q, K, V,
    attn_mask=None,
    dropout_p=0.0,
    is_causal=True,
)  # One line. Same output. ~3× faster on A100 for long sequences.

**Memory complexity comparison:**

| Method | Attention memory | Notes |
|---|---|---|
| Standard | $O(T^2)$ | Stores full $T \times T$ attention matrix to HBM |
| Flash Attention | $O(T)$ | Never materializes the full matrix |
| With KV cache | $O(T)$ | Cached K/V stored per layer |

: Attention memory comparison. {tbl-colwidths="[25, 20, 55]"}

For $T = 8192$: standard attention stores $8192^2 \times 6 \times 2 \approx 800$ MB of attention intermediates per forward pass. [Flash Attention stores essentially nothing.]{.mark}

## Speculative Decoding

KV cache and Flash Attention fix the per-token cost. Speculative decoding addresses a different problem: autoregressive generation is sequential — you cannot generate token $t+1$ until you have token $t$. On a GPU with thousands of cores, generating one token at a time wastes nearly all available parallelism.

**The idea:** use a small, fast **draft model** to propose $k$ tokens ahead. Then verify all $k$ proposals with the large **target model** in a single parallel forward pass. If the target agrees with the draft, all $k$ tokens are accepted at once. If it disagrees at position $j$, accept tokens up to $j-1$ and resample from the target at position $j$.

[The key insight: verification is cheap because all $k$ tokens are verified in a single batched forward pass]{.mark} — sampling from the large model $k$ times would require $k$ sequential forward passes.

### The acceptance criterion

Naive rejection (accept if the draft matches the target's greedy choice) changes the distribution. The **correct acceptance criterion** [@leviathan2023fast] preserves the exact target distribution:

Accept draft token $\tilde{x}$ at position $t$ with probability:

$$\alpha_t = \min\!\left(1,\; \frac{p_{\text{target}}(\tilde{x} \mid x_{<t})}{p_{\text{draft}}(\tilde{x} \mid x_{<t})}\right).$$

If rejected, sample a correction token from:

$$p_{\text{corrected}}(x) \propto \max\!\left(0,\; p_{\text{target}}(x) - p_{\text{draft}}(x)\right).$$

[This guarantees the output distribution is exactly $p_{\text{target}}$, regardless of draft model quality.]{.underline} A worse draft model just has lower acceptance rate; it never corrupts the distribution.

In [ ]:
def speculative_decode_step(
    target_logits: torch.Tensor,
    draft_logits: torch.Tensor,
    draft_tokens: torch.Tensor,
    temperature: float = 1.0,
) -> tuple[torch.Tensor, int]:
    """One step of speculative decoding: verify k draft tokens against the target.

    Args:
        target_logits: shape (1, k+1, V) — target model logits on the draft tokens
                       plus one extra position for the bonus token.
        draft_logits: shape (1, k, V) — draft model logits when sampling.
        draft_tokens: shape (1, k) — proposed token IDs from the draft.
        temperature: sampling temperature for probability computation.

    Returns:
        accepted_tokens: shape (1, n_accepted) — accepted token IDs.
        n_accepted: how many draft tokens were accepted (0 <= n_accepted <= k+1).
    """
    k = draft_tokens.size(1)

    target_probs = F.softmax(target_logits[:, :k, :] / max(temperature, 1e-6), dim=-1)  # (1, k, V)
    draft_probs = F.softmax(draft_logits / max(temperature, 1e-6), dim=-1)               # (1, k, V)

    accepted = []
    n_accepted = 0

    for i in range(k):
        token = draft_tokens[0, i].item()
        p_target = target_probs[0, i, token].item()
        p_draft = draft_probs[0, i, token].item()

        acceptance_prob = min(1.0, p_target / (p_draft + 1e-10))  # <1>

        if torch.rand(1).item() < acceptance_prob:
            accepted.append(token)
            n_accepted += 1
        else:
            # Sample correction from residual distribution
            residual = torch.clamp(target_probs[0, i] - draft_probs[0, i], min=0)  # <2>
            residual_sum = residual.sum()
            if residual_sum > 1e-8:
                residual = residual / residual_sum
                correction = torch.multinomial(residual, num_samples=1).item()
            else:
                correction = torch.multinomial(target_probs[0, i], num_samples=1).item()
            accepted.append(correction)
            n_accepted += 1
            break  # stop after first rejection  # <3>

    # Bonus token: sample from target at position after last accepted
    if n_accepted == k:
        bonus = torch.multinomial(
            F.softmax(target_logits[0, k, :] / max(temperature, 1e-6), dim=-1),
            num_samples=1,
        ).item()
        accepted.append(bonus)
        n_accepted += 1

    accepted_tensor = torch.tensor(accepted, dtype=torch.long).unsqueeze(0)
    return accepted_tensor, n_accepted

1. The acceptance probability $\min(1, p_T / p_D)$: if the target assigns more probability than the draft, always accept; if less, accept proportionally.
2. The residual distribution $\max(0, p_T - p_D)$ is the correction distribution — sampling from it on rejection ensures the overall distribution is exactly $p_T$.
3. Stop after the first rejection: subsequent positions are invalid because they were conditioned on the (incorrect) rejected token.

### Speedup analysis

The theoretical speedup depends on $k$ (draft tokens per step) and $\alpha$ (mean acceptance rate). Mean tokens accepted per target forward pass:

$$\bar{k} = \sum_{i=1}^{k} \alpha^i + \alpha^k.$$

For $\alpha = 0.8$ and $k = 4$:

$$\bar{k} = 0.8 + 0.64 + 0.51 + 0.41 + 0.41 \approx 2.77.$$

So you get ~2.77 tokens per target forward pass instead of 1. Accounting for the draft model overhead:

$$\text{speedup} \approx \frac{\bar{k}}{1 + \text{draft cost fraction} \times k}.$$

If the draft model costs 10% of the target per token and $k = 4$: speedup $\approx 2.77 / 1.4 \approx 2×$.

## The `FastInferenceEngine`

Combining KV cache, Flash Attention, and speculative decoding into a single coherent interface:

In [ ]:
import time


class FastInferenceEngine:
    """Combines KV cache, Flash Attention, and speculative decoding.

    Flash Attention is used automatically via F.scaled_dot_product_attention
    when the model's attention module is updated to use it. KV cache and
    speculative decoding are managed explicitly by this engine.

    Args:
        model: target language model.
        tokenizer: tokenizer with encode/decode methods.
        device: compute device.
        use_speculative: enable speculative decoding.
        draft_model: small draft model (required if use_speculative=True).
        k: number of draft tokens per step.
        max_cache_len: maximum cache sequence length before eviction.
        eviction_strategy: callable(cache, max_len) for cache eviction.
    """

    def __init__(
        self,
        model,
        tokenizer,
        device: torch.device,
        use_speculative: bool = False,
        draft_model=None,
        k: int = 4,
        max_cache_len: int = 2048,
        eviction_strategy=None,
    ) -> None:
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.use_spec = use_speculative and (draft_model is not None)
        self.draft_model = draft_model
        self.k = k
        self.max_cache = max_cache_len
        self.eviction = eviction_strategy or SlidingWindowEviction()

        if hasattr(F, "scaled_dot_product_attention"):
            print("Flash Attention available via F.scaled_dot_product_attention")
        else:
            print("Flash Attention unavailable — upgrade to PyTorch >= 2.0")

    def generate(
        self,
        prompt: str,
        max_new_tokens: int = 200,
        temperature: float = 0.8,
        top_k: int = 50,
    ) -> tuple[str, dict]:
        """Generate a response with performance statistics.

        Args:
            prompt: input prompt string.
            max_new_tokens: maximum tokens to generate.
            temperature: sampling temperature.
            top_k: top-k sampling parameter.

        Returns:
            (generated_text, stats) where stats contains 'tokens_per_sec',
            'n_tokens', 'elapsed_sec'.
        """
        t0 = time.perf_counter()

        if self.use_spec:
            text, n_tokens = self._speculative_generate(
                prompt, max_new_tokens, temperature, top_k
            )
        else:
            text, n_tokens = self._standard_generate(
                prompt, max_new_tokens, temperature, top_k
            )

        elapsed = time.perf_counter() - t0
        stats = {
            "tokens_per_sec": n_tokens / max(elapsed, 1e-6),
            "n_tokens": n_tokens,
            "elapsed_sec": elapsed,
        }
        return text, stats

    def _standard_generate(
        self, prompt: str, max_new_tokens: int, temperature: float, top_k: int
    ) -> tuple[str, int]:
        text = generate_with_cache(
            self.model, self.tokenizer, prompt,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            device=self.device,
        )
        return text, len(self.tokenizer.encode(text))

    def _speculative_generate(
        self, prompt: str, max_new_tokens: int, temperature: float, top_k: int
    ) -> tuple[str, int]:
        # Simplified: call speculative_decode_step in a loop
        # (full implementation would manage draft/target caches together)
        raise NotImplementedError(
            "Speculative generation requires model-level KV cache integration."
        )

    def benchmark(
        self,
        prompts: list[str],
        max_new_tokens: int = 100,
    ) -> dict:
        """Measure mean tokens/sec over a list of prompts.

        Args:
            prompts: list of prompt strings.
            max_new_tokens: tokens to generate per prompt.

        Returns:
            Dict with 'mean_tps', 'min_tps', 'max_tps'.
        """
        tps_list = []
        for p in prompts:
            _, stats = self.generate(p, max_new_tokens=max_new_tokens)
            tps_list.append(stats["tokens_per_sec"])
        return {
            "mean_tps": sum(tps_list) / len(tps_list),
            "min_tps": min(tps_list),
            "max_tps": max(tps_list),
        }

## Summary

| Concept | Key detail |
|---|---|
| Without KV cache | $O(t)$ work per token → $O(T^2)$ total; throughput collapses for long sequences |
| KV cache | Cache $K_l$, $V_l$ after computing them; $O(1)$ work per new token |
| KV cache memory | $2LHd_hT \times \text{bytes}$; linear in sequence length |
| Prefill vs decode | Prefill: full prompt in one pass. Decode: one token at a time using cache |
| Sliding window | Keep most recent $T_{\max}$ tokens; simple, loses distant context |
| Sink token | Keep first $n_\text{sink}$ + most recent; preserves attention anchors |
| Flash Attention bottleneck | Standard attention: $O(T^2)$ HBM reads/writes per layer — IO-bound |
| Online softmax | One-pass accumulation without storing the full attention matrix |
| Flash Attention memory | $O(T)$ vs $O(T^2)$; one line: `F.scaled_dot_product_attention` |
| Speculative decoding | Draft $k$ tokens with small model; verify in one target forward pass |
| Acceptance criterion | $\min(1, p_T / p_D)$; preserves exact target distribution |
| Correction sampling | Sample from $\max(0, p_T - p_D)$ on rejection |
| Speedup | Mean tokens per pass $= \sum_{i=1}^k \alpha^i + \alpha^k$; ~2.77× for $\alpha{=}0.8, k{=}4$ |

: Inference optimization reference. {tbl-colwidths="[30, 70]"}

## Exercises

1. **Cache memory profiler.** Run generation for sequences of lengths $\{64, 128, 256, 512, 1024\}$ and record peak GPU memory at each length with and without the KV cache. Plot memory vs sequence length for both cases. Confirm without cache: quadratic growth; with cache: linear.

2. **Distribution preservation.** Verify `speculative_decode_step` preserves the target distribution. Run 10,000 speculative decoding steps with a known target distribution. Plot the empirical output distribution and confirm it matches the target, not the draft.

3. **Online softmax verification.** Run `flash_attention_reference` and compare its output to standard scaled-dot-product attention on random inputs (B=2, H=4, T=128, d_h=64). Confirm the outputs match to within $10^{-4}$ tolerance.

4. **Acceptance rate vs speedup.** Simulate speculative decoding with acceptance rates $\alpha \in \{0.5, 0.7, 0.8, 0.9, 0.95\}$ and draft lengths $k \in \{2, 4, 8\}$. Plot the theoretical $\bar{k}$ (mean accepted tokens) as a function of $\alpha$ and $k$. At what $\alpha$ does $k = 8$ become better than $k = 4$?

5. **Sink token eviction.** Implement a benchmark that runs generation on a document longer than `max_cache_len`. Compare output quality (perplexity on the full document) between `SlidingWindowEviction` and `SinkTokenEviction`. When does the sink token strategy help most?

6. **Draft model selection.** Run speculative decoding with draft models of different sizes (e.g. a 2-layer vs 4-layer vs 6-layer version of NanoGPT). Measure acceptance rate $\alpha$ and actual tokens/sec for each. Find the Pareto-optimal draft model size (best speedup vs quality tradeoff).

■